# L2 — LSTM para estimar o SoC (PESI004) — formato de sequência

Rede LSTM (PyTorch + scikit-learn) que estima o estado de carga (SoC) a partir da **corrente e da tensão medidas**: as mesmas medições, a 1 s, que a Contagem de Coulomb (CC) e o EKF recebem no MATLAB. Como no artigo de Chemali et al. (2018), a rede **percorre o perfil inteiro desde t = 0**, carregando a memória de um segundo para o outro, e devolve o SoC **a cada segundo**, como a CC e o EKF. A avaliação usa os mesmos arquivos de teste da CC e do EKF (180 na versão 10) e a mesma régua (`metricas_erro.m`).

**Por que este formato (rodadas 1 e 2, 02/11):** no formato anterior, cada estimativa olhava só uma janela dos últimos segundos, com a memória zerada no começo da janela. Na faixa plana da OCV da LFP, a tensão quase não muda, e o SoC só pode ser acompanhado contando a carga desde um ponto conhecido. A janela jogava essa história fora: a rede aprendia o treino, mas errava 15–20% na validação. No formato de sequência, a memória vem desde o começo do perfil.

**Como rodar**
1. **Ambiente de execução → Alterar o tipo de ambiente de execução → T4 GPU → Salvar.**
2. Célula 1: `modo_rapido = True` → **Ambiente de execução → Executar tudo** (poucos minutos; só confere se roda; os números não valem).
3. Troque para `modo_rapido = False` → **Arquivo → Salvar** → **Executar tudo**. Deixe a aba aberta.
4. Se o Colab cair, use **Executar tudo** de novo: as redes já treinadas são lidas do Drive.

**Rodada 9 (04/11) — 10 sementes de treino, todo o resto fixo:** Vidal et al. (J. Energy Storage, 2022) recomendam repetir o treino pelo menos 10 vezes, porque o erro muda com os pesos iniciais. Decidido em 04/11, **antes de qualquer resultado com 10 sementes**: neurônios, taxa, dados, arquitetura e regras de treino continuam os da rodada 8 (128 neurônios, taxa 0,01, escolhidos pela validação); as redes das sementes 1 a 5 e os candidatos da escolha são **lidos** do Drive (`resultados_lstm_v10/modelos`) e só as sementes 6 a 10 são treinadas. O resultado passa a ser média ± desvio das 10 redes e o comitê de 10, **qualquer que seja o número**; a rodada 8 (5 sementes) fica guardada em `resultados_lstm_v10`.

**O que sai em `Meu Drive/LSTM_SoC/resultados_lstm_v10_10s` (rodada 9; as redes ficam em `resultados_lstm_v10/modelos`)**
- `metricas_MC_todos_LSTM.csv` e `metricas_MC_resumo_LSTM.csv`: mesmas colunas do `metricas_MC_todos.csv` e do `metricas_MC_resumo.csv` do MATLAB.
- `escolha_hiperparametros.csv` e `treino_sementes.csv`: escolhas pela validação e treino final.
- `previsoes/`: SoC estimado em cada arquivo de teste (figuras da L3 no MATLAB). `modelos/` e `figuras/`.

**Caminho:** dados → normalização → sequências → LSTM → treino com parada antecipada → escolha dos neurônios, da taxa e dos erros de sensor pela validação → 10 sementes (+ comitê) → teste.

**Rodada 7 (03/11, ~14h30) e versão 10 — perfis padrão de condução:** a versão 9 deu val100 4,4% (comitê) e val60 32,8% (rede não identifica o SoC inicial no patamar). Decisões de 03/11, tomadas antes de qualquer teste nos perfis novos: (1) o professor pediu testar os métodos em mais perfis de corrente além do HPPC → teste em **val100 + FUDS + US06** (ciclos padrão; corrente do banco público CALCE, célula A123 LFP, 20 °C; pico escalado para 2C, limite da célula na Tab. 5; pulsos de carga viram repouso, porque os parâmetros da referência foram identificados só em descarga); o val60 fica para a fase com dados reais; (2) **DST no treino** (divisão usual: Chen et al., 2023, Energy); (3) **mais dados**: 40 aleatórios novos de treino e 6 de validação, metade com degraus rápidos (validação de 4 arquivos estava instável: sementes de 5 a 14%). Arquitetura, grade e regras de escolha: as mesmas da versão 9. Resultados da rodada 7 guardados em `resultados_lstm_pura`.

**Rodada 6 (03/11) e versão 9 — LSTM pura no protocolo da literatura:** com a entrada $Q$ (versão 8), o teste não melhorou (comitê: val100 15,8%, val60 19,9%). A rede acerta a contagem, mas não descobre o SoC inicial pela tensão na faixa plana (no val60 ficou ~20% acima o tempo todo). Além disso, a versão 8 é híbrida (usa a capacidade $C_n$), e a comparação com a CC e o EKF pede uma rede **puramente de dados**, a categoria de IA da referência. A leitura dos artigos com LSTM pura em células LFP 18650 mostrou a diferença de protocolo: em Yang et al. (IEEE Access, 2019), a célula A123 18650 LFP é **carregada por completo** e o perfil é repetido **até a descarga completa**; treino e teste começam na carga completa, a 1 s, com I, V e T, e o SoC inicial desconhecido é avaliado à parte (testes começando em 80%, 60%, 40% e 20%). Chemali et al. (2018) também treinam com ciclos completos concatenados. Nas versões 7 e 8, o treino misturava SoC inicial sorteado e "inícios virtuais", e a rede passou a "ler a tensão" desde o começo (no val100, que começa cheio, caiu ~15 p.p. quando a carga entrou). **Versão 9:** só I e V; treino (24 arquivos: HPPC + 23 aleatórios) e validação (4) **só com descargas que começam na carga completa**; sem inícios virtuais; 32, 64 ou 128 neurônios (os artigos usam 50 a 500). No teste, o val100 corresponde ao caso de carga completa dos artigos e o val60, ao teste de SoC inicial desconhecido. Os resultados das versões 7 e 8 ficam guardados (`resultados_lstm` e `resultados_lstm_VIQ`).

**Rodada 5 (02/11, ~11h50) e versão 8:** com 44 perfis de treino, a rede deixou de decorar (treino e validação ficaram parecidos), mas parou em ~13–17% nos dois: ela **não consegue aprender sozinha a somar a corrente por milhares de segundos**. Algumas sementes chegaram perto (RMSE de teste de ~7–10%) e outras ficaram presas em ~16–17%, que é o que se consegue só "lendo a tensão". Na versão 8, a rede recebe também a **carga retirada desde o começo**, $Q$ (a mesma conta da CC, sem o SoC inicial). Assim ela não precisa descobrir a integração: precisa aprender **de onde partiu** (pela tensão do começo) e **corrigir** a conta com a tensão, que é o que o EKF faz com o modelo. Modelos assim, que juntam a contagem de carga com uma rede, são chamados híbridos na literatura. Os resultados só com I e V (rodada 5) ficam guardados em `resultados_lstm`, para comparação.

**Rodada 4 (02/11, ~10h45):** o treino desceu até ~6%, mas a validação ficou em 13–19% (a rede decorou os 3 perfis aleatórios de treino). Correção: 40 perfis extra de treino e 6 de validação (célula 3).

**Rodada 3 (02/11, ~10h30):** no formato de sequência, cada rede treinou em 3–5 s e parou com o RMSE de treino ainda em ~17–19%: paciência curta demais para épocas tão pequenas. A versão atual treina muito mais tempo (paciência de ~7200 atualizações) e deixa a validação decidir se os erros de sensor no treino ajudam.

## 1. Ajustes

Todos os números que podem mudar ficam **nesta célula**, como no `configuracao.m`.

In [ ]:
# ===== 1. AJUSTES (todos os números que podem mudar ficam AQUI, num lugar só) =====
# --- Versão 9 (03/11): LSTM PURA no protocolo da literatura (ver o texto antes da célula 1) ---
usar_carga_acumulada = False                                 # False = só I e V (LSTM pura, como Chemali 2018 e Yang 2019); True = I, V e Q (versão 8, híbrida, só para comparação)
Cn_Ah = 1.462                                                # Capacidade da célula (Ah) usada em Q (só na versão 8): a MESMA da CC e do EKF (parametros_bat.m, Tabela 6)
apenas_inicio_carga_completa = True                          # True = treino e validação só com descargas que começam na carga completa, como nos artigos (Yang 2019: carga completa e descarga até o fim)
soc_ini_carga_completa = 0.995                               # SoC inicial mínimo (fração) para contar como "carga completa"

pasta_base = '/content/drive/MyDrive/LSTM_SoC'              # Pasta do projeto no Google Drive
pasta_dados = pasta_base + '/dados_lstm'                     # Os .csv exportados pelo gerar_dados_lstm.m (MATLAB)
pasta_modelos_reuso = None                                   # None = redes na subpasta modelos da pasta de resultados
if usar_carga_acumulada:                                     # Versão 8 (híbrida)
    pasta_res = pasta_base + '/resultados_lstm_VIQ'          # Resultados da rodada 6
elif apenas_inicio_carga_completa:                           # Versões 9 e 10 (pura, protocolo da literatura)
    pasta_res = pasta_base + '/resultados_lstm_v10_10s'      # Rodada 9 (04/11): versão 10 com 10 sementes; a rodada 8 (5 sementes) fica em resultados_lstm_v10
    pasta_modelos_reuso = pasta_base + '/resultados_lstm_v10/modelos'   # Redes já treinadas na rodada 8 (escolha + sementes 1 a 5): lidas, não treinadas de novo; as novas (6 a 10) são gravadas junto
else:                                                        # Versão 7 (pura, com SoC inicial sorteado no treino)
    pasta_res = pasta_base + '/resultados_lstm'              # Resultados da rodada 5

modo_rapido = False                                          # True = teste curto só para ver se tudo roda (números NÃO valem); False = rodada de verdade

# --- Entradas e saída (mesmos nomes das colunas dos .csv do MATLAB) ---
colunas_entrada = ['I_medido', 'Vt_medido']                  # Entradas: corrente e tensão MEDIDAS (as mesmas da CC e do EKF)
coluna_saida = 'Soc'                                         # Saída: SoC de referência (fração de 0 a 1)
perfis_teste = ['val100_p3', 'fuds_p3', 'us06_p3']          # Versão 10: perfis de TESTE (iguais ao lista_perfis_teste do configuracao.m). O val60 fica para os dados reais
n_teste_esperado = len(perfis_teste) * 3 * 20                # 3 perfis × 3 cenários de sensor × 20 sorteios = 180
Ts_med = 1                                                   # Período das amostras (s), igual ao Ts_med do configuracao.m

# --- Formato de sequência (Chemali et al., 2018) ---
passo_tbptt = 1000                                           # Trecho do treino (passos): a memória passa de um trecho ao outro; o gradiente volta até 1000 passos (Chemali: 1000)
dur_max_sequencia_s = 14400                                  # Duração máxima de cada sequência de treino/validação (s): 4 h cobre as descargas completas inteiras (DST ~3,3 h; aleatórios 1-3 h); só o HPPC (15 h) é cortado
tamanho_lote_seq = 8                                         # Sequências por lote

# --- Inícios virtuais (BMS ligado depois de um repouso, em qualquer SoC) ---
usar_inicios_virtuais = False                                # Versão 9: desligado (nos artigos, cada sequência começa no início real da descarga). True = também "começa" no fim de cada repouso longo (versões 7 e 8)
dur_min_repouso_s = 300                                      # Repouso mínimo (s) para virar um começo
limiar_repouso_A = 0.1                                       # |corrente| abaixo disso = repouso (o ruído de base tem desvio de ~15 mA)

# --- Erros de sensor no TREINO (aumento de dados; validação e teste NÃO mudam) ---
usar_erros_sensor = False                                    # Etapa C (treino com erros de sensor). Desligada: nas rodadas 4 e 5, o treino limpo venceu na validação (9,2 × 10,1% e 12,2 × 14,9%)
lista_fracoes_com_erro = [0.0, 0.5]                          # Candidatos da etapa C: fração das sequências de treino com erro (0 = treino limpo; 0,5 = metade com erro)
fracao_base = 0.0                                            # Fração usada nas etapas A e B (aprender primeiro a tarefa principal, com treino limpo)
desvio_max_I_A = 0.110                                       # Desvio (offset) de corrente sorteado entre -0,110 e +0,110 A (Vidal et al., 2020)
desvio_max_V_V = 0.004                                       # Desvio de tensão sorteado entre -4 e +4 mV (Vidal et al., 2020)
ganho_max_I = 0.02                                           # Erro de ganho da corrente sorteado entre -2% e +2% (ordem dos usados em McMaster)

# --- Rede (escolhida SÓ pela validação) ---
lista_n_ocultos = [32, 64, 128]                              # Candidatos de unidades da camada LSTM (etapa A). Artigos: 50 por camada (Yang 2019) e 500 (Chemali 2018); a célula 6 confere a regra "mais padrões que pesos" (aula 9)
lista_taxas_aprendizado = [3e-4, 1e-3, 3e-3, 1e-2]          # Candidatos de taxa de aprendizado do Adam (etapa B). Rodada 3: 3e-3 venceu na borda → 1e-2 incluída
taxa_base = 1e-3                                             # Taxa usada na etapa A (valor padrão do Adam)
vies_esquecimento = 1.0                                      # Viés inicial da porta de esquecimento (1 = começa "lembrando"; Jozefowicz et al., 2015)

# --- Treino ---
fator_reducao_taxa = 0.5                                     # Redução da taxa: multiplica por 0,5 quando a validação para de melhorar
reducao_atualizacoes = 2400                                  # ... depois de ~2400 atualizações seguidas sem melhora (convertido em épocas na 1ª época)
paciencia_atualizacoes = 7200                                # Parada antecipada: ~7200 atualizações seguidas sem melhora (rodada 4: suficiente para o treino descer)
max_atualizacoes = 100000                                    # Limite de atualizações (folga; o notebook avisa se a melhor época ficar perto dele)
mostrar_a_cada = 20                                          # Mostra o progresso a cada 20 épocas
limite_gradiente = 1.0                                       # Corte da norma do gradiente (evita passos enormes em sequências longas)
semente_escolha = 1                                          # Semente de treino usada na escolha (célula 8)
sementes_treino = list(range(1, 11))                         # 10 sementes de treino da rede final (resultado = média ± desvio); Vidal et al. (2022): repetir o treino pelo menos 10 vezes (decidido em 04/11, antes de ver o resultado)

# --- Avaliação (IGUAL ao configuracao.m, seção 5) ---
limiar_conv = 0.02                                           # Faixa de convergência (fração): ±2 pontos percentuais
t_inicio_avaliacao_min = 0                                   # Métricas desde t = 0 (min), como na CC e no EKF

# --- Identificação nas tabelas (mesmas colunas do MATLAB) ---
fonte_dados = 'simulacao'                                    # Dados simulados nesta fase
modo_config = 'AF'                                           # Configuração da apresentação final
nome_metodo = 'LSTM (V, I, Q)' if usar_carga_acumulada else 'LSTM'   # Nome do método na coluna 'metodo'
if usar_carga_acumulada and usar_erros_sensor:               # Os erros de sensor teriam de mudar também a carga Q (não implementado)
    raise ValueError('usar_erros_sensor só funciona com usar_carga_acumulada = False')   # Para com a explicação

if modo_rapido:                                              # Ajustes do teste curto (só confere se o código roda)
    lista_n_ocultos = [16, 32]                               # Dois tamanhos (para testar a etapa A)
    lista_taxas_aprendizado = [1e-3, 3e-3]                   # Duas taxas (para testar a etapa B)
    lista_fracoes_com_erro = [0.0, 0.5]                      # As duas frações (para testar a etapa C)
    mostrar_a_cada = 1                                       # Mostra todas as épocas
    sementes_treino = [1, 2]                                 # Duas sementes (para testar o comitê)
    max_epocas_rapido = 3                                    # Três épocas
    pasta_res = pasta_base + '/resultados_lstm_teste_rapido' # Pasta separada: não mistura com os resultados de verdade
    pasta_modelos_reuso = None                               # No teste rápido, as redes também ficam na pasta separada
    print('MODO RÁPIDO: só para conferir se tudo roda. Os números NÃO valem.')   # Aviso

## 2. Bibliotecas, Google Drive e GPU

- **NumPy** e **pandas**: vetores e tabelas (o papel das matrizes e do `readtable` do MATLAB).
- **PyTorch** (`torch`): monta a LSTM, calcula as derivadas sozinho (*autograd*) e roda na GPU.
- **scikit-learn**: só o normalizador (`StandardScaler`).
- **Matplotlib**: gráficos de conferência.

In [ ]:
# ===== 2. BIBLIOTECAS, DRIVE E GPU =====
import os, glob, re, time, copy, random                     # Utilitários: pastas, busca de arquivos, texto, tempo, cópia, sorteio
import numpy as np                                           # Vetores e matrizes (como no MATLAB)
import pandas as pd                                          # Tabelas: leitura e gravação dos .csv
import matplotlib.pyplot as plt                              # Gráficos
import torch                                                 # PyTorch: tensores na GPU e derivadas automáticas
import torch.nn as nn                                        # Camadas prontas (LSTM, Linear)
import sklearn                                               # scikit-learn (aqui só para mostrar a versão)
from sklearn.preprocessing import StandardScaler             # Normalizador z-score do scikit-learn

try:                                                         # No Colab: conecta o Google Drive
    from google.colab import drive                           # Módulo do Colab que acessa o Drive
    drive.mount('/content/drive')                            # Na primeira vez, pede autorização da sua conta
except ImportError:                                          # Fora do Colab (ex.: no PC), usa as pastas como estão
    print('Fora do Colab: usando as pastas dos ajustes sem montar o Drive')   # Aviso

dispositivo = torch.device('cuda' if torch.cuda.is_available() else 'cpu')   # Usa a GPU se houver; senão, a CPU
pasta_modelos = pasta_modelos_reuso if pasta_modelos_reuso else os.path.join(pasta_res, 'modelos')   # Redes treinadas (.pt); na rodada 9, a pasta da rodada 8 (reuso)
print(f'Redes (.pt) em: {pasta_modelos}')                    # Mostra de onde as redes são lidas e onde as novas são gravadas
pasta_previsoes = os.path.join(pasta_res, 'previsoes')       # SoC estimado em cada arquivo de teste
pasta_figuras = os.path.join(pasta_res, 'figuras')           # Figuras de conferência (.png)
for pasta in [pasta_res, pasta_modelos, pasta_previsoes, pasta_figuras]:   # Cria as pastas que faltarem
    os.makedirs(pasta, exist_ok=True)                        # exist_ok=True: não dá erro se a pasta já existir

print(f'PyTorch {torch.__version__} | scikit-learn {sklearn.__version__} | NumPy {np.__version__} | pandas {pd.__version__}')   # Versões (vão para o artigo)
print(f'Dispositivo: {dispositivo}' + (f' ({torch.cuda.get_device_name(0)})' if dispositivo.type == 'cuda' else ''))   # Mostra se está na GPU
if dispositivo.type == 'cpu':                                # Sem GPU o treino fica muito lento
    print('ATENÇÃO: sem GPU. Ambiente de execução → Alterar o tipo de ambiente de execução → T4 GPU.')   # Como ligar a GPU

## 3. Dados: treino, validação e teste

| Papel | Arquivos | Para quê |
|---|---|---|
| **Treino** | HPPC + 43 perfis aleatórios (s11–s13 + 40 extra, s101–s140) | A rede ajusta os pesos olhando só para eles |
| **Validação** | 7 perfis aleatórios (s21 + 6 extra, s201–s206) | Decide quando parar o treino e quais neurônios, taxa e erros de sensor usar |

**Perfis extra (02/11):** com só 3 perfis aleatórios, a rede decorou o treino (rodada 4: RMSE de treino ~6%, validação 13–19%). Os 40 perfis extra de treino (20 começando em 100% e 20 num SoC sorteado entre 20% e 100%, em repouso) e os 6 de validação dão a diversidade que falta: com dezenas de perfis diferentes, a única regra que serve para todos é contar a carga e corrigir com a tensão. São ~200 mil amostras de treino, na mesma ordem dos conjuntos de ciclos concatenados usados por Chemali et al. (2018). Gerados pelo `gerar_dados_lstm_extra.m` (MATLAB).
| **Teste** | val100, FUDS e US06 × 3 cenários × 20 sorteios = 180 (versão 10; até a versão 9: val100 e val60) | Nota final. A rede nunca viu esses arquivos |

**Versão 10:** entram 40 perfis aleatórios novos de treino (20 com degraus de 30 a 600 s e 20 com degraus rápidos de 2 a 60 s, s141–s180), 6 de validação (s207–s212) e o **DST** (só descarga) no treino, todos começando cheios: **65 de treino e 10 de validação**. Teste: val100 + **FUDS** e **US06** (só descarga). Gerados pelo `gerar_dados_lstm_v10.m`.

**Versão 9:** saem do treino e da validação os arquivos com SoC inicial sorteado (s121–s140 e s204–s206). Ficam 24 de treino (HPPC, s11–s13, s101–s120) e 4 de validação (s21, s201–s203), todos com carga completa no início e descarga até ~5%, como os ciclos dos artigos (Yang et al., 2019; Chemali et al., 2018).

Os arquivos de teste são **as mesmas medições** que a CC e o EKF receberam no `rodar_cenarios.m`. A célula confere cada arquivo: colunas certas, nenhum valor vazio e passo de 1 s.

In [ ]:
# ===== 3. LEITURA DOS DADOS =====
arquivos = sorted(glob.glob(os.path.join(pasta_dados, 'dados_lstm_*.csv')))           # Todos os .csv da pasta, em ordem alfabética
arq_treino = [a for a in arquivos if os.path.basename(a).startswith('dados_lstm_treino_')]        # Arquivos de treino
arq_validacao = [a for a in arquivos if os.path.basename(a).startswith('dados_lstm_validacao_')]  # Arquivo(s) de validação
arq_teste = [a for a in arquivos if os.path.basename(a).startswith('dados_lstm_teste_')]          # Arquivos de teste
arq_teste = [a for a in arq_teste if any(os.path.basename(a).startswith(f'dados_lstm_teste_{p}_') for p in perfis_teste)]   # Só os perfis de teste da versão 10 (o val60 fica de fora)
if modo_rapido:                                              # No teste curto, só o sorteio r1 de cada combinação
    arq_teste = [a for a in arq_teste if '_r1_' in os.path.basename(a)]   # 9 arquivos em vez de 180
assert arq_treino and arq_validacao and arq_teste, f'Faltam arquivos em {pasta_dados}'   # Para tudo se alguma lista vier vazia

padrao_teste = re.compile(r'dados_lstm_teste_(?P<perfil>.+?_p(?P<ordem>\d+))_(?P<cenario>.+)_r(?P<semente>\d+)_[\d.]+s\.csv$')   # Nome: perfil, ordem da planta, cenário e sorteio
dados = {}                                                   # Dicionário: caminho do arquivo → tabela
for arq in arq_treino + arq_validacao + arq_teste:           # Lê cada arquivo uma vez só
    tabela = pd.read_csv(arq)                                # Lê o .csv (vírgula como separador, ponto decimal)
    faltando = [c for c in ['t_min'] + colunas_entrada + [coluna_saida] if c not in tabela.columns]   # Colunas que deveriam existir
    assert not faltando, f'{os.path.basename(arq)}: faltam as colunas {faltando}'                     # Para se faltar coluna
    assert not tabela.isna().any().any(), f'{os.path.basename(arq)}: tem valores vazios (NaN)'         # Para se houver vazio
    passo_s = np.diff(tabela['t_min'].to_numpy()) * 60       # Intervalo entre amostras (min → s)
    assert np.allclose(passo_s, Ts_med, atol=1e-3), f'{os.path.basename(arq)}: passo diferente de {Ts_med} s'   # Confere o passo
    dados[arq] = tabela                                      # Guarda a tabela

info_teste = []                                              # Uma linha por arquivo de teste (perfil, cenário, sorteio)
for arq in arq_teste:                                        # Lê as informações do nome de cada arquivo de teste
    m = padrao_teste.match(os.path.basename(arq))            # Aplica o padrão ao nome
    assert m is not None, f'Nome fora do padrão: {os.path.basename(arq)}'   # Para se o nome não seguir o padrão
    info_teste.append({'arquivo': arq, 'perfil': m['perfil'], 'ordem_planta': int(m['ordem']),
                       'cenario_sensor': m['cenario'], 'semente': int(m['semente'])})   # Guarda as informações
info_teste = pd.DataFrame(info_teste)                        # Vira tabela

def resumo_arquivo(arq):                                     # Resumo de um arquivo para conferência
    t = dados[arq]                                           # Tabela do arquivo
    return {'arquivo': os.path.basename(arq), 'amostras': len(t),                                   # Nome e número de amostras
            'SoC_ini_%': round(100 * t[coluna_saida].iloc[0], 1), 'SoC_fim_%': round(100 * t[coluna_saida].iloc[-1], 1),   # SoC no início e no fim
            'I_max_A': round(t['I_medido'].max(), 2), 'repouso_%': round(100 * (t['I_medido'].abs() < limiar_repouso_A).mean())}   # Corrente máxima e tempo em repouso

print(pd.DataFrame([resumo_arquivo(a) for a in arq_treino + arq_validacao]).to_string(index=False))   # Tabela de treino e validação
print(f'\nTreino: {len(arq_treino)} arquivos | Validação: {len(arq_validacao)} | Teste: {len(arq_teste)}')   # Contagem
print(info_teste.groupby(['perfil', 'cenario_sensor']).size().rename('sorteios').to_string())         # Sorteios por combinação de teste
if not modo_rapido and len(arq_teste) != n_teste_esperado:   # O esperado é 3 perfis × 3 cenários × 20 sorteios
    print(f'ATENÇÃO: eram esperados {n_teste_esperado} arquivos de teste e há {len(arq_teste)} (faltam os arquivos do gerar_dados_lstm_v10.m?)')   # Aviso
if len(arq_treino) < 44 or len(arq_validacao) < 7:           # O esperado é 44 de treino (HPPC + 3 + 40 extra) e 7 de validação (1 + 6 extra)
    print('ATENÇÃO: faltam os perfis extra (gerar_dados_lstm_extra.m no MATLAB → arrastar para dados_lstm no Drive)')   # Aviso

if apenas_inicio_carga_completa:                             # Versão 9: protocolo dos artigos (descargas que começam na carga completa)
    def comeca_cheio(arq):                                   # True se o arquivo começa na carga completa
        return dados[arq][coluna_saida].iloc[0] >= soc_ini_carga_completa   # SoC de referência no 1º instante
    fora = [os.path.basename(a) for a in arq_treino + arq_validacao if not comeca_cheio(a)]   # Arquivos que saem (SoC inicial sorteado)
    arq_treino = [a for a in arq_treino if comeca_cheio(a)]  # Fica só quem começa cheio
    arq_validacao = [a for a in arq_validacao if comeca_cheio(a)]   # Idem na validação
    assert arq_treino and arq_validacao, 'Nenhum arquivo começa na carga completa'   # Para se a lista ficar vazia
    print(f'\nVersões 9/10: ficam {len(arq_treino)} de treino e {len(arq_validacao)} de validação, todos começando na carga completa.')   # Resumo
    if len(arq_treino) != 65 or len(arq_validacao) != 10:    # Versão 10: 24 antigos + 40 novos + DST; 4 antigos + 6 novos
        print('ATENÇÃO: a versão 10 espera 65 de treino e 10 de validação (faltam os arquivos do gerar_dados_lstm_v10.m no Drive?)')   # Aviso
    print(f'Saem {len(fora)} arquivos com SoC inicial sorteado: {fora}')   # Quais saíram

## 4. Normalização (só com o treino)

Corrente (0 a 2,8 A) e tensão (2,3 a 3,6 V) têm escalas diferentes. O **z-score** deixa cada entrada com média 0 e desvio 1:

$$x_{norm} = \frac{x - \mu}{\sigma}$$

**Versão 9 (atual): só I e V**, sem entrada extra. O texto abaixo vale só para a versão 8 (`usar_carga_acumulada = True`).

**Entrada extra $Q$ (versão 8):** a carga retirada desde o começo da sequência, em fração da capacidade,

$$Q_k = \frac{T_s}{C_n}\,\big(I_0 + I_1 + \dots + I_{k-1}\big), \qquad Q_0 = 0,$$

com $C_n$ = 1,462 Ah, **a mesma capacidade da CC e do EKF**. É a conta da CC sem o SoC inicial: $SoC_k = SoC_0 - Q_k$. A rede continua sem receber o SoC inicial; ela precisa estimá-lo pela tensão e corrigir a conta ao longo do tempo. Se o sensor de corrente tiver desvio (bias), $Q$ herda o erro, como a CC; a tensão é o que pode corrigir.

**$\mu$ e $\sigma$ são calculados só com os arquivos de treino** e aplicados, sem mudar, na validação e no teste. Se o teste entrasse nessa conta, uma informação do teste "vazaria" para a rede (*data leakage*). A saída (SoC, de 0 a 1) não precisa ser normalizada.

In [ ]:
# ===== 4. NORMALIZAÇÃO (z-score, ajustada SÓ no treino) =====
nomes_entradas = colunas_entrada + (['Q_retirada'] if usar_carga_acumulada else [])   # Nomes das entradas da rede

def entradas_brutas(tabela, c=0, fim=None):
    """Entradas de um trecho do arquivo (da amostra c até fim), sem normalizar: I, V e, se pedido, Q.
    Q = carga retirada desde o COMEÇO do trecho, em fração da capacidade (mesma conta da CC, sem o SoC inicial):
    Q_k = (Ts/Cn)·(I_0 + ... + I_(k-1)), com Q_0 = 0. Só usa amostras passadas."""
    t = tabela.iloc[c:fim]                                   # Trecho do arquivo
    x = t[colunas_entrada].to_numpy(dtype=np.float64)        # I e V medidos
    if usar_carga_acumulada:                                 # Acrescenta a carga retirada
        I = t['I_medido'].to_numpy(dtype=np.float64)         # Corrente medida (A, descarga positiva)
        Q = np.concatenate([[0.0], np.cumsum(I[:-1])]) * Ts_med / 3600 / Cn_Ah   # Soma da corrente ANTES de cada instante (A·s → fração de Cn)
        x = np.column_stack([x, Q])                          # Terceira coluna
    return x

x_treino_todos = np.vstack([entradas_brutas(dados[a]) for a in arq_treino])   # Entradas de todos os arquivos de treino (Q desde o começo de cada arquivo)
normalizador = StandardScaler().fit(x_treino_todos)          # Calcula média e desvio de cada entrada (só treino)

def entradas(tabela, c=0, fim=None):                         # Entradas normalizadas com a MESMA média e o MESMO desvio
    return normalizador.transform(entradas_brutas(tabela, c, fim)).astype(np.float32)   # (x - média)/desvio, em float32 (formato da GPU)

for nome, media, desvio in zip(nomes_entradas, normalizador.mean_, normalizador.scale_):  # Mostra os valores usados
    print(f'{nome}: média = {media:.4f} | desvio = {desvio:.4f}')   # Uma linha por entrada

## 5. Sequências

A rede lê cada perfil **do começo ao fim**, um segundo por vez, e a memória de um segundo passa para o próximo:

$$(I, V)_0 \to (I, V)_1 \to \dots \to (I, V)_k \;\Longrightarrow\; \widehat{SoC}_0,\ \widehat{SoC}_1,\ \dots,\ \widehat{SoC}_k$$

É o mesmo jeito da CC e do EKF: cada estimativa usa **tudo o que veio antes**, nunca o futuro. Por isso a rede pode aprender a "contar carga" na memória e a corrigir essa conta com a tensão.

- **Começos:** a memória começa em zero no início do perfil (t = 0), como no teste.
- **Inícios virtuais:** um BMS pode ser ligado em qualquer SoC, mas todos os arquivos de treino começam em 100%. Por isso, no treino e na validação, o fim de cada repouso longo (≥ 5 min) também vira um começo, com a memória zerada ali, como uma bateria ligada depois de descansar. O HPPC tem repousos de 1 h a cada 5–10% de SoC, o que dá começos de 95% a 5%. A escolha vem da aplicação, não do teste; o teste não muda.
- **Duração máxima de 2 h por sequência:** cobre com folga a validação (78 min) e os testes (23 e 36 min). O HPPC (15 h) fica coberto pelos seus vários começos virtuais, sem repetir 15 h de dados a cada época.
- **Equilíbrio entre arquivos:** o HPPC tem mais sequências e mais passos que os aleatórios, e fica ~93% do tempo em repouso. No cálculo do erro, cada passo recebe um peso tal que **cada arquivo de treino** (com seus começos virtuais) pesa 25% do total.
- **Sequências de tamanhos diferentes** são completadas com zeros no fim, e esses passos têm peso zero (uma "máscara" marca os passos válidos).

In [ ]:
# ===== 5. SEQUÊNCIAS =====
def achar_inicios_virtuais(tabela):
    """Índices da ÚLTIMA amostra de cada repouso longo (>= dur_min_repouso_s) que é seguido de carga."""
    repouso = (tabela['I_medido'].abs() < limiar_repouso_A).to_numpy()   # True onde a corrente é ~0
    inicios, k, n = [], 0, len(repouso)                       # Lista de inícios, posição atual, tamanho
    while k < n:                                             # Percorre o arquivo
        if not repouso[k]:                                   # Fora de repouso: segue
            k += 1                                           # Próxima amostra
            continue                                         # Volta ao início do laço
        fim_rep = k                                          # Começo de um repouso
        while fim_rep < n and repouso[fim_rep]:              # Anda até o fim do repouso
            fim_rep += 1                                     # fim_rep = 1ª amostra depois do repouso
        if k > 0 and fim_rep < n and (fim_rep - k) * Ts_med >= dur_min_repouso_s:   # Não é o começo do arquivo, tem carga depois e é longo
            inicios.append(fim_rep - 1)                      # Última amostra do repouso
        k = fim_rep                                          # Continua depois do repouso
    return inicios                                           # Lista de índices

def montar_sequencias(lista_arquivos, inicios_virtuais=False, dur_max_s=None):
    """Uma sequência por começo (t = 0 do arquivo e, se pedido, o fim de cada repouso longo), até o fim do arquivo (no máximo dur_max_s).
    Saídas: X (seq × passos × entradas), Y (seq × passos), M (máscara: 1 = passo válido), id_arquivo e comprimento de cada sequência."""
    pedacos = []                                             # (entradas, SoC, arquivo, SoC no começo) de cada sequência
    for i_arq, arq in enumerate(lista_arquivos):             # Um arquivo por vez
        y = dados[arq][coluna_saida].to_numpy().astype(np.float32)   # SoC de referência
        comecos = [0] + (achar_inicios_virtuais(dados[arq]) if inicios_virtuais else [])   # Começo real + virtuais
        for c in comecos:                                    # Uma sequência por começo
            fim = len(y) if dur_max_s is None else min(len(y), c + int(round(dur_max_s / Ts_med)))   # Fim da sequência
            pedacos.append((entradas(dados[arq], c, fim), y[c:fim], i_arq))   # Entradas normalizadas (Q recomeça em zero em cada começo)
    n_seq, n_passos = len(pedacos), max(len(p[1]) for p in pedacos)   # Quantas sequências e a maior duração
    X = np.zeros((n_seq, n_passos, len(nomes_entradas)), dtype=np.float32)    # Entradas, completadas com zero no fim
    Y = np.zeros((n_seq, n_passos), dtype=np.float32)        # SoC, completado com zero no fim
    M = np.zeros((n_seq, n_passos), dtype=np.float32)        # Máscara: 1 nos passos válidos, 0 no enchimento
    for k, (xs, ys, _) in enumerate(pedacos):                # Copia cada sequência para a sua linha
        X[k, :len(ys)], Y[k, :len(ys)], M[k, :len(ys)] = xs, ys, 1.0   # Passos válidos
    id_arquivo = np.array([p[2] for p in pedacos])           # De qual arquivo veio cada sequência
    comprimento = np.array([len(p[1]) for p in pedacos])     # Passos válidos de cada sequência
    return X, Y, M, id_arquivo, comprimento

def pesos_equilibrados(M, id_arquivo):
    """Peso de cada passo: cada arquivo (com seus começos virtuais) fica com o mesmo peso total; média 1 por passo válido."""
    n_arq = id_arquivo.max() + 1                             # Número de arquivos
    passos_arq = np.array([M[id_arquivo == i].sum() for i in range(n_arq)])   # Passos válidos de cada arquivo
    W = M / passos_arq[id_arquivo][:, None]                  # Arquivo com muitos passos → peso menor por passo
    return (W * M.sum() / W.sum()).astype(np.float32)        # Escala para média 1 por passo válido

X_demo, Y_demo, M_demo, id_demo, comp_demo = montar_sequencias(arq_treino, usar_inicios_virtuais, dur_max_sequencia_s)   # Sequências de treino
W_demo = pesos_equilibrados(M_demo, id_demo)                 # Pesos
print(f'Sequências de treino: {len(Y_demo)} | maior: {comp_demo.max()} passos | lotes por época: {int(np.ceil(len(Y_demo) / tamanho_lote_seq))}')   # Resumo
for i_arq, arq in enumerate(arq_treino):                     # Por arquivo: começos, passos e peso
    sel = id_demo == i_arq                                   # Sequências deste arquivo
    socs = [round(100 * float(Y_demo[k, 0])) for k in np.flatnonzero(sel)]   # SoC no começo de cada sequência
    print(f'{os.path.basename(arq):58s} passos: {M_demo[sel].sum() / M_demo.sum():6.1%} | peso: {W_demo[sel].sum() / W_demo.sum():6.1%} | começos (SoC %): {socs}')   # Fração
X_dv, Y_dv, _, _, _ = montar_sequencias(arq_validacao, usar_inicios_virtuais, dur_max_sequencia_s)   # Sequências de validação
print(f'Validação: {len(Y_dv)} sequências, começos (SoC %): {[round(100 * float(Y_dv[k, 0])) for k in range(len(Y_dv))]}')   # Resumo

## 6. A rede LSTM

A LSTM lê a sequência **um segundo por vez** e carrega uma memória de um passo para o outro. Essa memória tem duas partes: o **estado da célula** $c_t$ (memória longa) e a **saída oculta** $h_t$. Três **portas**, com valores entre 0 e 1 (sigmoide $\sigma$), decidem o que entra e o que sai da memória:

$$
\begin{aligned}
f_t &= \sigma(W_f\,x_t + U_f\,h_{t-1} + b_f) &&\text{porta de esquecimento: quanto da memória antiga fica}\\
i_t &= \sigma(W_i\,x_t + U_i\,h_{t-1} + b_i) &&\text{porta de entrada: quanto da informação nova entra}\\
\tilde c_t &= \tanh(W_c\,x_t + U_c\,h_{t-1} + b_c) &&\text{informação nova candidata}\\
c_t &= f_t \odot c_{t-1} + i_t \odot \tilde c_t &&\text{memória atualizada}\\
o_t &= \sigma(W_o\,x_t + U_o\,h_{t-1} + b_o) &&\text{porta de saída: quanto da memória aparece}\\
h_t &= o_t \odot \tanh(c_t), \qquad \widehat{SoC}_t = w^\top h_t + b
\end{aligned}
$$

Aqui $x_t = (I_t, V_t)$ normalizados e $\odot$ é a multiplicação elemento a elemento.

**Ligação com o problema (e por que a entrada $Q$ ajuda):** com $f_t \approx 1$, a memória vai **somando** o que entra a cada passo ($c_t \approx c_{t-1} + i_t \odot \tilde c_t$). É a mesma conta da CC, $SoC_{k+1} = SoC_k - \tfrac{\Delta t\,\eta}{C_n} I_k$. Em tese, a LSTM pode aprender essa soma sozinha, mas, na rodada 5, ela não conseguiu por milhares de segundos com os dados disponíveis. Com a entrada $Q$, a soma já vem pronta, e a rede se concentra no que a CC não faz: descobrir o SoC inicial pela tensão e corrigir a conta.

**Arquitetura (como em Chemali et al., 2018):** 1 camada LSTM (16, 32 ou 64 unidades, escolhido pela validação) → uma camada linear que transforma $h_t$ no SoC **a cada passo**. A memória começa em zero em t = 0.

**Viés da porta de esquecimento = 1.** Com $b_f = 0$, a porta começa em $\sigma(0) = 0{,}5$: a cada passo, metade da memória se perde. Com $b_f = 1$, ela começa em $\sigma(1) \approx 0{,}73$, e a rede começa o treino "lembrando" por mais tempo. Jozefowicz et al. (2015) recomendam essa prática, que é também o padrão do Keras. Só muda o ponto de partida: o treino ajusta o valor depois.

In [ ]:
# ===== 6. A REDE LSTM =====
class RedeLSTM(nn.Module):                                   # Toda rede do PyTorch herda de nn.Module
    def __init__(self, n_entradas, n_ocultos):               # Define as camadas (e cria os pesos com valores aleatórios)
        super().__init__()                                   # Inicializa a parte do nn.Module
        self.lstm = nn.LSTM(input_size=n_entradas, hidden_size=n_ocultos, num_layers=1, batch_first=True)   # 1 camada LSTM; batch_first: (sequências, passos, entradas)
        self.saida = nn.Linear(n_ocultos, 1)                 # Camada linear: h de cada passo → SoC
        with torch.no_grad():                                # Ajuste do valor inicial (não é treino)
            self.lstm.bias_ih_l0[n_ocultos:2 * n_ocultos].fill_(vies_esquecimento)   # Viés da porta de esquecimento (2º bloco: ordem i, f, g, o)
            self.lstm.bias_hh_l0[n_ocultos:2 * n_ocultos].fill_(0.0)   # O PyTorch soma dois vieses; o 2º fica 0 para o total ser vies_esquecimento

    def forward(self, x, estado=None):                       # Conta feita quando se chama modelo(x, estado)
        seq_h, estado = self.lstm(x, estado)                 # h de todos os passos e a memória (h, c) do último passo
        return self.saida(seq_h).squeeze(-1), estado         # SoC a cada passo (sequências × passos) e a memória para continuar depois

n_amostras_treino = sum(len(dados[a]) for a in arq_treino)   # Padrões de treino (instantes com SoC conhecido)
for n_oc in lista_n_ocultos:                                 # Tamanho de cada rede candidata
    n_parametros = sum(p.numel() for p in RedeLSTM(len(nomes_entradas), n_oc).parameters())   # Conta os pesos da rede
    print(f'LSTM({len(nomes_entradas)} → {n_oc}) + Linear({n_oc} → 1): {n_parametros:6d} pesos | '
          f'padrões de treino / pesos = {n_amostras_treino / n_parametros:5.1f}')   # Regra da aula 9: mais padrões que pesos

## 7. Treino com parada antecipada

**O que é treinar:** ajustar os pesos para que o SoC estimado fique perto do SoC de referência **em todos os passos**. O erro é o **erro quadrático médio ponderado** (MSE), o quadrado do RMSE usado na avaliação, com os pesos da célula 5.

**Retropropagação truncada no tempo (TBPTT, como Chemali et al., 2018):** cada lote tem 8 sequências de até 2 h. A rede percorre o lote em **trechos de 1000 s**:
1. estima o SoC de cada passo do trecho, começando com a memória que veio do trecho anterior (*forward*);
2. calcula o erro do trecho;
3. `perda.backward()` calcula o gradiente, que volta passo a passo **dentro do trecho**;
4. o gradiente muito grande é cortado (norma máxima 1);
5. o **Adam** atualiza os pesos;
6. a memória (h, c) **continua** para o próximo trecho, mas o caminho do gradiente é cortado ali (`detach`). A rede lembra do passado inteiro, e o gradiente volta no máximo 1000 passos (custo limitado).

**Época:** uma passada por todas as sequências de treino, em ordem sorteada.

**Erros de sensor no treino (aumento de dados, etapa C da célula 8):** no teste, a corrente medida tem desvio (bias) e ruído, como num sensor real. O EKF já está preparado pelas matrizes Q e R; a CC não tem preparação; a LSTM não tem Q nem R, e o jeito de "avisar" que o sensor erra é mostrar exemplos no treino: uma fração das sequências recebe um erro sorteado, o mesmo no perfil inteiro, como um sensor descalibrado (desvio de corrente de até ±110 mA, desvio de tensão de até ±4 mV, Vidal et al., 2020, grupo de McMaster, e ganho de até ±2% na corrente). No formato de sequência, porém, um desvio constante de corrente atrapalha a contagem de carga que a rede precisa aprender. Por isso, as etapas A e B usam treino limpo, e a etapa C compara treino limpo × metade das sequências com erro: **a validação decide**. **A validação e o teste não mudam** em nenhum caso; a robustez a erros de sensor é sempre medida nos cenários de teste da referência.

**Redução da taxa:** se a validação passar ~2400 atualizações sem melhorar, a taxa cai pela metade (passos grandes no começo, refinamento no fim).

**Parada antecipada:** se a validação passar ~7200 atualizações sem melhorar, o treino para, e ficam **os pesos da melhor época**. A paciência é contada em atualizações e convertida em épocas no fim da 1ª época (o notebook mostra a conversão), porque o tamanho da época muda com a quantidade de dados. Na rodada 3, com paciência curta, o treino parou com a rede ainda aprendendo; na rodada 4, com ~7200 atualizações, o treino desceu até ~6%. O limite de 100 mil atualizações é só uma folga; o notebook avisa se a melhor época ficar perto dele.

**Como saber se o treino foi adequado:** o RMSE de **treino** na melhor época (mostrado no fim de cada rede e na tabela da célula 8) deve ficar baixo. Se ele continuar alto, a rede não aprendeu nem os dados que viu, e o problema não é generalização.

**Sementes:** os pesos iniciais, a ordem das sequências e os erros sorteados dependem de uma semente. Fixar a semente torna o treino repetível; 10 sementes (Vidal et al., 2022: pelo menos 10 repetições) mostram quanto o resultado depende da sorte do início.

**Retomada:** cada rede treinada é salva no Drive (`modelos/lstmseq_H<neurônios>_lr<taxa>_e<fração com erro>_s<semente>.pt`) e é lida em vez de treinada outra vez se os ajustes forem os mesmos.

In [ ]:
# ===== 7. FUNÇÕES DE TREINO E PREVISÃO =====
torch.backends.cudnn.deterministic = True                    # Contas da GPU repetíveis (mesma semente → mesmo resultado)
torch.backends.cudnn.benchmark = False                       # Não troca de algoritmo entre execuções (também para repetir)
i_I, i_V = nomes_entradas.index('I_medido'), nomes_entradas.index('Vt_medido')   # Posição da corrente e da tensão nas entradas

def fixar_semente(semente):                                  # Fixa todas as fontes de números aleatórios
    random.seed(semente)                                     # Python
    np.random.seed(semente)                                  # NumPy
    torch.manual_seed(semente)                               # PyTorch (pesos iniciais)
    torch.cuda.manual_seed_all(semente)                      # PyTorch na GPU

@torch.no_grad()                                             # Na previsão não há treino: desliga o cálculo de gradientes
def prever_seq(modelo, X_t):                                 # SoC estimado em todos os passos de todas as sequências
    modelo.eval()                                            # Modo de avaliação
    saida, _ = modelo(X_t.contiguous())                      # Uma passada do começo ao fim, memória começando em zero
    return saida.cpu().numpy()                               # Volta para a CPU como NumPy (sequências × passos)

def rmse_mascara(Y_est, Y, M):                               # RMSE (%) só nos passos válidos
    return float(np.sqrt(np.sum(M * (Y_est - Y) ** 2) / np.sum(M)) * 100)   # Raiz da média dos erros ao quadrado × 100

def erros_sensor_lote(n, sorteador, fracao_com_erro):
    """Sorteia um erro de sensor para cada uma das n sequências (o mesmo erro no perfil inteiro, como um sensor real).
    Só uma fração (fracao_com_erro) das sequências recebe erro; as outras ficam limpas.
    Em unidades físicas: I' = I·(1 + ganho) + desvio_I e V' = V + desvio_V. Na escala normalizada: x' = x·mult + soma."""
    com_erro = sorteador.random(n) < fracao_com_erro               # Quais sequências recebem erro
    ganho = sorteador.uniform(-ganho_max_I, ganho_max_I, n) * com_erro         # Erro de ganho da corrente
    desvio_I = sorteador.uniform(-desvio_max_I_A, desvio_max_I_A, n) * com_erro   # Desvio da corrente (A)
    desvio_V = sorteador.uniform(-desvio_max_V_V, desvio_max_V_V, n) * com_erro   # Desvio da tensão (V)
    mult = np.ones((n, len(nomes_entradas)), dtype=np.float32)     # Fator que multiplica cada entrada (1 = sem erro)
    soma = np.zeros((n, len(nomes_entradas)), dtype=np.float32)    # Valor somado a cada entrada (0 = sem erro)
    mult[:, i_I] = 1 + ganho                                       # Ganho na corrente
    soma[:, i_I] = (normalizador.mean_[i_I] * ganho + desvio_I) / normalizador.scale_[i_I]   # Ganho e desvio da corrente, na escala normalizada
    soma[:, i_V] = desvio_V / normalizador.scale_[i_V]             # Desvio da tensão, na escala normalizada
    return torch.from_numpy(mult).to(dispositivo), torch.from_numpy(soma).to(dispositivo)   # Para a GPU

def ajustes_treino():                                        # Tudo que muda a rede treinada (para saber se o arquivo salvo ainda vale)
    return {'formato': 'sequencia', 'nomes_entradas': list(nomes_entradas), 'Cn_Ah': Cn_Ah, 'passo_tbptt': passo_tbptt,
            'dur_max_sequencia_s': dur_max_sequencia_s, 'tamanho_lote_seq': tamanho_lote_seq,
            'usar_inicios_virtuais': usar_inicios_virtuais, 'dur_min_repouso_s': dur_min_repouso_s, 'limiar_repouso_A': limiar_repouso_A,
            'desvio_max_I_A': desvio_max_I_A,
            'desvio_max_V_V': desvio_max_V_V, 'ganho_max_I': ganho_max_I, 'vies_esquecimento': vies_esquecimento,
            'fator_reducao_taxa': fator_reducao_taxa, 'reducao_atualizacoes': reducao_atualizacoes,
            'paciencia_atualizacoes': paciencia_atualizacoes, 'max_atualizacoes': max_atualizacoes, 'limite_gradiente': limite_gradiente,
            'arq_treino': [os.path.basename(a) for a in arq_treino], 'arq_validacao': [os.path.basename(a) for a in arq_validacao]}

def treinar(n_ocultos, taxa_aprendizado, fracao_com_erro, semente):
    """Treina uma rede (unidades, taxa, fração de sequências com erro de sensor e semente dadas) ou lê do Drive, se já foi treinada com os mesmos ajustes."""
    nome = f'lstmseq_H{n_ocultos}_lr{taxa_aprendizado:g}_e{fracao_com_erro:g}_s{semente}'   # Nome da rede
    arquivo_modelo = os.path.join(pasta_modelos, nome + '.pt')   # Arquivo da rede no Drive
    if os.path.exists(arquivo_modelo):                       # Já foi treinada antes?
        salvo = torch.load(arquivo_modelo, map_location=dispositivo, weights_only=False)   # Lê o arquivo (fomos nós que gravamos)
        if salvo['ajustes'] == ajustes_treino() and np.allclose(salvo['media_norm'], normalizador.mean_):   # Mesmos ajustes e mesma normalização?
            modelo = RedeLSTM(len(nomes_entradas), n_ocultos).to(dispositivo)    # Cria a rede vazia
            modelo.load_state_dict(salvo['estado'])          # Coloca os pesos salvos
            print(f'{nome}: lida do Drive (RMSE de validação = {salvo["rmse_val"]:.3f}%, época {salvo["melhor_epoca"]})')   # Aviso
            return modelo, pd.DataFrame(salvo['historico']), salvo['rmse_val'], salvo['melhor_epoca']   # Devolve sem treinar
        print(f'{nome}: arquivo salvo com outros ajustes → treina de novo')   # Ajustes mudaram: treina outra vez

    fixar_semente(semente)                                   # Fixa a sorte ANTES de criar a rede (pesos iniciais dependem dela)
    modelo = RedeLSTM(len(nomes_entradas), n_ocultos).to(dispositivo)    # Rede com pesos iniciais aleatórios, na GPU
    X_tr, Y_tr, M_tr, id_tr, comp_tr = montar_sequencias(arq_treino, usar_inicios_virtuais, dur_max_sequencia_s)   # Sequências de treino
    W_tr = pesos_equilibrados(M_tr, id_tr)                   # Pesos (cada arquivo = 25%)
    X_va, Y_va, M_va, _, _ = montar_sequencias(arq_validacao, usar_inicios_virtuais, dur_max_sequencia_s)   # Sequências de validação
    X_tr_t, Y_tr_t, W_tr_t = (torch.from_numpy(v).to(dispositivo) for v in (X_tr, Y_tr, W_tr))   # Treino na GPU
    X_va_t = torch.from_numpy(X_va).to(dispositivo)          # Validação na GPU
    otimizador = torch.optim.Adam(modelo.parameters(), lr=taxa_aprendizado)   # Otimizador Adam
    redutor_taxa = torch.optim.lr_scheduler.ReduceLROnPlateau(otimizador, mode='min', factor=fator_reducao_taxa,
                                                              patience=10)   # Corta a taxa quando a validação estaciona (paciência ajustada no fim da 1ª época)
    sorteador = np.random.default_rng(semente)               # Gerador da ordem das sequências e dos erros de sensor (repetível)

    melhor_rmse, melhor_epoca, sem_melhora = np.inf, 0, 0    # Melhor validação até agora, época dela e contador de paciência
    melhor_estado, historico = None, []                      # Pesos da melhor época e curva de treino
    t0 = time.time()                                         # Marca o tempo
    epoca, paciencia_ep, max_ep = 0, None, None              # Época atual; paciência e limite em épocas (calculados na 1ª época)
    while True:                                              # Laço das épocas (sai pela parada antecipada ou pelo limite)
        epoca += 1                                           # Próxima época
        n_atualizacoes = 0                                   # Atualizações dos pesos nesta época
        modelo.train()                                       # Modo de treino
        ordem = sorteador.permutation(len(Y_tr))             # Ordem sorteada das sequências nesta época
        soma_erro2, soma_peso = 0.0, 0.0                     # Para o RMSE de treino da época
        for i in range(0, len(ordem), tamanho_lote_seq):     # Laço dos lotes de sequências
            sel_np = ordem[i:i + tamanho_lote_seq]           # Sequências deste lote
            sel = torch.from_numpy(sel_np).to(dispositivo)   # Índices na GPU
            x, y, w = X_tr_t[sel], Y_tr_t[sel], W_tr_t[sel]  # Entradas, SoC e pesos do lote
            if fracao_com_erro > 0:                          # Aumento de dados: erro de sensor sorteado por sequência
                mult, soma = erros_sensor_lote(len(sel_np), sorteador, fracao_com_erro)   # Um erro por sequência (só numa fração delas)
                x = x * mult[:, None, :] + soma[:, None, :]  # O mesmo erro em todos os passos da sequência
            peso_lote = float(w.sum().item())                # Peso total do lote (para a média ponderada)
            n_passos_lote = int(comp_tr[sel_np].max())       # Passos até o fim da sequência mais longa do lote
            estado = None                                    # Memória zerada no começo das sequências
            for t_ini in range(0, n_passos_lote, passo_tbptt):   # Laço dos trechos (TBPTT)
                trecho = slice(t_ini, min(t_ini + passo_tbptt, n_passos_lote))   # Passos deste trecho
                saida, estado = modelo(x[:, trecho].contiguous(), estado)   # 1. estima o SoC no trecho, continuando a memória (contiguous: trecho contínuo na memória da GPU)
                estado = tuple(e.detach() for e in estado)   # 6. a memória continua, mas o gradiente não volta além do trecho
                erro2_pond = w[:, trecho] * (saida - y[:, trecho]) ** 2   # Erro ao quadrado × peso de cada passo
                perda = erro2_pond.sum() / peso_lote         # 2. parcela do MSE ponderado do lote
                otimizador.zero_grad()                       # zera o gradiente do trecho anterior
                perda.backward()                             # 3. gradiente (retropropagação dentro do trecho)
                nn.utils.clip_grad_norm_(modelo.parameters(), max_norm=limite_gradiente)   # 4. corta gradiente muito grande
                otimizador.step()                            # 5. Adam atualiza os pesos
                n_atualizacoes += 1                          # Conta a atualização
                soma_erro2 += float(erro2_pond.sum().item()) # Acumula o erro do trecho
            soma_peso += peso_lote                           # Acumula o peso do lote
        if epoca == 1:                                       # Converte as paciências de atualizações para épocas (servem para qualquer volume de dados)
            if modo_rapido:                                  # Teste curto
                paciencia_ep, max_ep, redutor_taxa.patience = 2, max_epocas_rapido, 1
            else:                                            # Rodada de verdade
                paciencia_ep = max(10, int(np.ceil(paciencia_atualizacoes / n_atualizacoes)))   # Épocas sem melhora até parar
                redutor_taxa.patience = max(3, int(np.ceil(reducao_atualizacoes / n_atualizacoes)))   # Épocas sem melhora até reduzir a taxa
                max_ep = int(np.ceil(max_atualizacoes / n_atualizacoes))   # Limite de épocas
            print(f'  {nome}: 1 época = {n_atualizacoes} atualizações → paciência de {paciencia_ep} épocas, '
                  f'redução da taxa após {redutor_taxa.patience}, limite de {max_ep}')   # Mostra a conversão
        rmse_treino = float(np.sqrt(soma_erro2 / soma_peso) * 100)   # RMSE de treino da época (%, com erros de sensor)
        rmse_val = rmse_mascara(prever_seq(modelo, X_va_t), Y_va, M_va)   # RMSE de validação (%, sem erros, desde o começo)
        taxa_atual = otimizador.param_groups[0]['lr']        # Taxa usada nesta época
        historico.append({'epoca': epoca, 'rmse_treino_pct': rmse_treino, 'rmse_val_pct': rmse_val, 'taxa': taxa_atual})   # Guarda a curva
        redutor_taxa.step(rmse_val)                          # Se a validação não melhora há 'paciencia_reducao' épocas, reduz a taxa
        if rmse_val < melhor_rmse:                           # Melhorou a validação?
            melhor_rmse, melhor_epoca, sem_melhora = rmse_val, epoca, 0   # Nova melhor época; zera a paciência
            melhor_estado = copy.deepcopy(modelo.state_dict())   # Guarda uma cópia dos pesos desta época
        else:                                                # Não melhorou
            sem_melhora += 1                                 # Gasta um pouco da paciência
        if epoca == 1 or epoca % mostrar_a_cada == 0:        # Mostra a 1ª época e uma a cada 'mostrar_a_cada'
            print(f'  {nome} | época {epoca:3d} | taxa {taxa_atual:.1e} | RMSE treino {rmse_treino:7.3f}% | validação {rmse_val:7.3f}%'
                  + (' ← melhor' if sem_melhora == 0 else ''))   # Progresso
        if sem_melhora >= paciencia_ep:                      # Acabou a paciência
            print(f'  parada antecipada na época {epoca}: {paciencia_ep} épocas sem melhora')   # Aviso
            break                                            # Sai do laço das épocas
        if epoca >= max_ep:                                  # Chegou ao limite
            print(f'  limite de {max_ep} épocas atingido')   # Aviso
            break                                            # Sai do laço das épocas

    modelo.load_state_dict(melhor_estado)                    # Fica com os pesos da MELHOR época
    if not modo_rapido and melhor_epoca > max_ep - paciencia_ep:   # Ainda estava melhorando quando o limite acabou?
        print(f'  ATENÇÃO: {nome} ainda melhorava perto do limite de {max_ep} épocas → aumentar max_atualizacoes e rodar de novo')   # Aviso
    torch.save({'estado': melhor_estado, 'historico': historico, 'rmse_val': melhor_rmse, 'melhor_epoca': melhor_epoca,
                'n_ocultos': n_ocultos, 'taxa_aprendizado': taxa_aprendizado, 'fracao_com_erro': fracao_com_erro, 'semente': semente, 'ajustes': ajustes_treino(),
                'media_norm': normalizador.mean_.tolist(), 'desvio_norm': normalizador.scale_.tolist()}, arquivo_modelo)   # Salva no Drive
    print(f'{nome}: melhor validação = {melhor_rmse:.3f}% na época {melhor_epoca} de {epoca} | RMSE de treino nessa época = '
          f'{historico[melhor_epoca - 1]["rmse_treino_pct"]:.3f}% ({time.time() - t0:.0f} s)')   # Resumo
    return modelo, pd.DataFrame(historico), melhor_rmse, melhor_epoca   # Rede, curva, melhor RMSE e melhor época

## Boas práticas da aula 9 (Prof. Wilian) e onde estão neste notebook

| Dica da aula | Neste notebook |
|---|---|
| Normalizar os dados na faixa de ativação da rede | Z-score (célula 4): entradas com média 0 e desvio 1 |
| Sinal desejado um pouco abaixo do limite (0,9 e não 1) | Não precisa: a saída é **linear**, e não uma sigmoide que nunca chega a 1 |
| Não-linearidade tanh | Já está dentro da LSTM (tanh na memória; sigmoide nas portas) |
| Somar 0,05 na derivada | Truque da MLP clássica com sigmoide saturada; aqui, o Adam e a soma na memória da LSTM fazem esse papel |
| Taxa de aprendizado maior perto da entrada | O Adam usa um passo próprio para cada peso; a taxa inicial é escolhida pela validação e cai pela metade quando a validação estaciona |
| Mudar a ordem dos padrões a cada época | A ordem das sequências é sorteada a cada época |
| Pesos iniciais na região linear | Padrão do PyTorch (pesos pequenos); exceção: viés da porta de esquecimento = 1 |
| Métodos de aprendizado mais sofisticados | Adam |
| **Mais padrões de treino que pesos** | Razão mostrada na célula 6. Poucos neurônios → *underfitting*; muitos → *overfitting*. Por isso o número de neurônios é **escolhido pela validação** |
| Validação para parar o treino | Parada antecipada (~7200 atualizações sem melhora) |
| Treinar várias vezes | 10 sementes de treino (média ± desvio; Vidal et al., 2022) |
| Processamento paralelo | GPU |
| Eliminação de pesos (*weight decay*) | Não usado agora: a parada antecipada já regulariza |
| **Comitê de redes** | A média das 10 redes é avaliada como `LSTM (comitê de 10)` (célula 10), sem treino extra |

## 8. Escolha dos neurônios, da taxa e dos erros de sensor (só pela validação)

Pelo **menor RMSE na validação** (semente 1), em três etapas. O teste não entra nessa escolha.
- **Etapa A (neurônios):** taxa 0,001 e treino limpo; redes de 16, 32 e 64 neurônios (128 teria mais pesos que padrões de treino, regra da aula 9).
- **Etapa B (taxa):** com os neurônios escolhidos, taxas de 0,0003, 0,001, 0,003 e 0,01 (a de 0,001 já foi treinada e é só lida do Drive).
- **Etapa C (erros de sensor):** com neurônios e taxa escolhidos, treino limpo × metade das sequências com erro de sensor.

O trecho de 1000 s e a duração máxima de 2 h são fixos (Chemali et al., 2018, e a duração da validação e dos testes).

In [ ]:
# ===== 8. ESCOLHA DOS NEURÔNIOS, DA TAXA E DOS ERROS DE SENSOR (só validação) =====
historicos = {}                                              # Curvas de treino: (neurônios, taxa, fração, semente) → tabela
linhas_escolha = []                                          # Uma linha por rede treinada na escolha

def rodar_candidato(etapa, n_oc, taxa, fracao):              # Treina (ou lê) um candidato e anota o resultado
    modelo, hist, rmse_val, epoca = treinar(n_oc, taxa, fracao, semente_escolha)   # Semente da escolha
    historicos[(n_oc, taxa, fracao, semente_escolha)] = hist # Guarda a curva
    linhas_escolha.append({'etapa': etapa, 'n_ocultos': n_oc, 'taxa_aprendizado': taxa, 'fracao_com_erro': fracao,
                           'semente': semente_escolha, 'rmse_val_pct': rmse_val, 'melhor_epoca': epoca,
                           'rmse_treino_pct': float(hist.loc[epoca - 1, 'rmse_treino_pct'])})   # Resultado

def melhor_da_etapa(etapa, coluna):                          # Valor da coluna na linha de menor RMSE de validação da etapa
    tabela = pd.DataFrame([l for l in linhas_escolha if l['etapa'] == etapa])   # Linhas da etapa
    return tabela, tabela.loc[tabela['rmse_val_pct'].idxmin(), coluna]          # Tabela e valor escolhido

for n_oc in lista_n_ocultos:                                 # Etapa A: neurônios (taxa base, treino limpo)
    rodar_candidato('A_neuronios', n_oc, taxa_base, fracao_base)
tabela_neuronios, n_ocultos_escolhido = melhor_da_etapa('A_neuronios', 'n_ocultos')
n_ocultos_escolhido = int(n_ocultos_escolhido)               # Número inteiro de neurônios

for taxa in lista_taxas_aprendizado:                         # Etapa B: taxa (neurônios escolhidos, treino limpo)
    rodar_candidato('B_taxa', n_ocultos_escolhido, taxa, fracao_base)
tabela_taxas, taxa_escolhida = melhor_da_etapa('B_taxa', 'taxa_aprendizado')
taxa_escolhida = float(taxa_escolhida)                       # Número real

lista_c = lista_fracoes_com_erro if usar_erros_sensor else [fracao_base]   # Etapa C só se os erros de sensor estiverem ligados
for fracao in lista_c:                                       # Etapa C: treino limpo × metade com erros de sensor
    rodar_candidato('C_erros_sensor', n_ocultos_escolhido, taxa_escolhida, fracao)
tabela_fracoes, fracao_escolhida = melhor_da_etapa('C_erros_sensor', 'fracao_com_erro')
fracao_escolhida = float(fracao_escolhida)                   # Número real

tabela_escolha = pd.DataFrame(linhas_escolha)                # As três etapas juntas
tabela_escolha.to_csv(os.path.join(pasta_res, 'escolha_hiperparametros.csv'), index=False)   # Salva no Drive
print(tabela_escolha.round(4).to_string(index=False))        # Mostra a tabela
print(f'\nEscolhidos pela validação: {n_ocultos_escolhido} neurônios, taxa {taxa_escolhida:g}, fração com erro de sensor {fracao_escolhida:g}')   # Resultado

## 9. Rede final: 10 sementes de treino

Com os neurônios, a taxa e a fração de erros de sensor escolhidos, a rede é treinada com 10 sementes. Na rodada 9, a semente 1 (célula 8) e as sementes 2 a 5 (rodada 8) são **lidas** do Drive; só as sementes 6 a 10 são treinadas. Todas são testadas (média ± desvio) e também usadas juntas, como comitê. No fim, a célula confere se as sementes 1 a 5 lidas têm a mesma validação da rodada 8.

In [ ]:
# ===== 9. REDE FINAL COM 10 SEMENTES =====
modelos = {}                                                 # semente → rede treinada
linhas_sementes = []                                         # Uma linha por semente
for semente in sementes_treino:                              # Cada semente de treino
    modelo, hist, rmse_val, epoca = treinar(n_ocultos_escolhido, taxa_escolhida, fracao_escolhida, semente)   # Treina (ou lê)
    modelos[semente] = modelo                                # Guarda a rede
    historicos[(n_ocultos_escolhido, taxa_escolhida, fracao_escolhida, semente)] = hist   # Guarda a curva
    linhas_sementes.append({'n_ocultos': n_ocultos_escolhido, 'taxa_aprendizado': taxa_escolhida, 'fracao_com_erro': fracao_escolhida,
                            'semente_treino': semente, 'rmse_val_pct': rmse_val, 'melhor_epoca': epoca,
                            'rmse_treino_pct': float(hist.loc[epoca - 1, 'rmse_treino_pct'])})   # Resultado na validação
tabela_sementes = pd.DataFrame(linhas_sementes)              # Tabela do treino final
tabela_sementes.to_csv(os.path.join(pasta_res, 'treino_sementes.csv'), index=False)   # Salva no Drive
print(tabela_sementes.round(4).to_string(index=False))       # Mostra
print(f'\nValidação: RMSE = {tabela_sementes.rmse_val_pct.mean():.3f} ± {tabela_sementes.rmse_val_pct.std():.3f}% ({len(sementes_treino)} sementes)')   # Média ± desvio
arq_rodada8 = pasta_base + '/resultados_lstm_v10/treino_sementes.csv'   # Validação das 5 sementes da rodada 8
if pasta_modelos_reuso and os.path.exists(arq_rodada8):      # Rodada 9: confere se as redes 1 a 5 foram lidas (e não treinadas de novo)
    antigas = pd.read_csv(arq_rodada8).set_index('semente_treino')['rmse_val_pct']   # RMSE de validação da rodada 8, por semente
    novas = tabela_sementes.set_index('semente_treino')['rmse_val_pct']   # RMSE de validação desta rodada, por semente
    comuns = antigas.index.intersection(novas.index)         # Sementes presentes nas duas (1 a 5)
    dif = (antigas[comuns] - novas[comuns]).abs().max()      # Maior diferença (p.p.)
    print(f'Conferência com a rodada 8 (sementes {list(comuns)}): maior diferença de validação = {dif:.2e} p.p.')   # Deve dar 0
    if dif > 1e-6:                                           # Diferente: as redes foram treinadas de novo (ajustes mudaram)
        print('ATENÇÃO: as sementes 1 a 5 não são as da rodada 8. Pare e avise o Claude antes de usar o resultado.')

## 10. Teste: mesma régua da CC e do EKF

Cada rede (10 sementes) percorre cada arquivo de teste (180) **do começo ao fim, uma vez só**, com a memória começando em zero em t = 0, e devolve o SoC a cada segundo, como a CC e o EKF. As métricas usam a função `metricas_erro` copiada do `metricas_erro.m`:
- **RMSE** (p.p.): métrica da referência;
- **MAE** (p.p.): métrica de Chemali et al.;
- **erro máximo** (p.p.);
- **tempo de convergência** (min): quando o erro entra em ±2 p.p. e não sai mais (NaN = terminou fora da faixa).

**Colunas iguais às do MATLAB**, com duas observações: `Soc0_est_pct` fica vazio (a LSTM não recebe SoC inicial, então o mesmo resultado vale para as duas colunas da tabela final) e `frac_dentro_3sigma` fica vazio (só o EKF calcula essa incerteza).

**Dois métodos na tabela:** `LSTM` (cada rede sozinha: 20 sorteios × 10 sementes = 200 linhas por caso) e `LSTM (comitê de 10)` (média das 10 redes: 20 linhas por caso).

**Tempo de execução:** medido na **CPU** (o MATLAB também roda na CPU): o perfil inteiro com a rede da semente 1, mediana de 3 repetições, em um arquivo de cada perfil. No comitê, o tempo é multiplicado pelo número de redes.

In [ ]:
# ===== 10. TESTE NOS ARQUIVOS DE TESTE =====
def metricas_erro(Soc_est, Soc_ref, t_min, limiar_conv):
    """Cópia fiel do metricas_erro.m: a MESMA régua para CC, EKF e LSTM. Erros em pontos percentuais (p.p.), tempo em min."""
    erro = (Soc_est - Soc_ref) * 100                         # Erro em p.p.
    rmse = float(np.sqrt(np.mean(erro ** 2)))                # RMSE
    mae = float(np.mean(np.abs(erro)))                       # MAE
    erro_max = float(np.max(np.abs(erro)))                   # Erro máximo
    fora = np.flatnonzero(np.abs(erro) > limiar_conv * 100)  # Instantes FORA da faixa de ±limiar
    if fora.size == 0:                                       # Nunca saiu da faixa
        t_conv = 0.0                                         # Converge desde o início
    elif fora[-1] == len(erro) - 1:                          # Ainda fora no último instante
        t_conv = np.nan                                      # Não convergiu dentro do ensaio
    else:                                                    # Saiu da faixa e voltou de vez
        t_conv = float(t_min[fora[-1] + 1] - t_min[0])       # Tempo até entrar de vez na faixa
    return rmse, mae, erro_max, t_conv                       # Mesmas saídas do MATLAB

def entradas_arquivo(arq, disp):                             # Perfil inteiro de um arquivo, pronto para a rede (1 × passos × 2)
    return torch.from_numpy(entradas(dados[arq])[None]).to(disp)     # [None] cria a dimensão de "sequências" (uma só); Q começa em zero em t = 0

# --- Tempo de execução na CPU (um arquivo por perfil) ---
modelo_cpu = copy.deepcopy(modelos[sementes_treino[0]]).to('cpu')   # Cópia da rede da 1ª semente na CPU
tempo_por_perfil = {}                                        # perfil → tempo (s)
for perfil, grupo in info_teste.groupby('perfil'):           # Um perfil por vez
    arq_tempo = grupo.sort_values(['cenario_sensor', 'semente']).iloc[0]['arquivo']   # 1º arquivo do perfil (nominal, r1)
    x_cpu = entradas_arquivo(arq_tempo, 'cpu')               # Entradas na CPU
    tempos = []                                              # Tempo de cada repetição
    for repeticao in range(3):                               # 3 repetições
        t0 = time.perf_counter()                             # Relógio de alta resolução
        prever_seq(modelo_cpu, x_cpu)                        # Estima o perfil inteiro
        tempos.append(time.perf_counter() - t0)              # Tempo gasto
    tempo_por_perfil[perfil] = float(np.median(tempos))      # Mediana das 3
    print(f'Tempo na CPU, {perfil}: {tempo_por_perfil[perfil]:.3f} s para {len(dados[arq_tempo])} amostras')   # Mostra

# --- Avaliação: todos os arquivos de teste × todas as sementes (+ comitê) ---
nome_comite = f'{nome_metodo} (comitê de {len(sementes_treino)})'   # Nome do método do comitê
linhas = []                                                  # Uma linha por arquivo × semente de treino (+ comitê)
for n, info in enumerate(info_teste.itertuples(index=False), start=1):   # Um arquivo de teste por vez
    tabela = dados[info.arquivo]                             # Tabela do arquivo
    x_t = entradas_arquivo(info.arquivo, dispositivo)        # Perfil inteiro na GPU
    y = tabela[coluna_saida].to_numpy()                      # SoC de referência
    t_min = tabela['t_min'].to_numpy()                       # Tempo (min)
    idx_aval = t_min >= t_inicio_avaliacao_min               # Instantes avaliados (iguais aos da CC e do EKF)
    previsoes = pd.DataFrame({'t_min': t_min, 'Soc': y})     # Tabela de previsões deste arquivo
    for semente, modelo in modelos.items():                  # Cada rede treinada
        Soc_lstm = prever_seq(modelo, x_t)[0]                # SoC estimado a cada segundo (uma passada desde t = 0)
        previsoes[f'Soc_lstm_s{semente}'] = Soc_lstm         # Guarda a previsão desta semente
        rmse, mae, erro_max, t_conv = metricas_erro(Soc_lstm[idx_aval], y[idx_aval], t_min[idx_aval], limiar_conv)   # Mesma régua
        linhas.append({'perfil': info.perfil, 'fonte': fonte_dados, 'modo': modo_config, 'ordem_planta': info.ordem_planta,
                       'cenario_sensor': info.cenario_sensor, 'Ts_med_s': Ts_med, 'Soc0_real_pct': round(100 * float(y[0])),
                       'Soc0_est_pct': np.nan, 'metodo': nome_metodo, 'rmse_pct': rmse, 'mae_pct': mae, 'erro_max_pct': erro_max,
                       't_conv_min': t_conv, 'tempo_exec_s': tempo_por_perfil[info.perfil], 'frac_dentro_3sigma': np.nan,
                       't_inicio_aval_min': t_inicio_avaliacao_min, 'semente': info.semente,
                       'convergiu': int(not np.isnan(t_conv)), 'semente_treino': semente})   # Mesmas colunas do metricas_MC_todos.csv (+ semente_treino)
    previsoes['Soc_lstm_media'] = previsoes[[f'Soc_lstm_s{s}' for s in modelos]].mean(axis=1)   # Média das sementes = comitê
    if len(modelos) > 1:                                     # Comitê só faz sentido com mais de uma rede
        Soc_comite = previsoes['Soc_lstm_media'].to_numpy()  # Estimativa do comitê
        rmse, mae, erro_max, t_conv = metricas_erro(Soc_comite[idx_aval], y[idx_aval], t_min[idx_aval], limiar_conv)   # Mesma régua
        linhas.append({**linhas[-1], 'metodo': nome_comite, 'rmse_pct': rmse, 'mae_pct': mae, 'erro_max_pct': erro_max,
                       't_conv_min': t_conv, 'tempo_exec_s': tempo_por_perfil[info.perfil] * len(modelos),
                       'convergiu': int(not np.isnan(t_conv)), 'semente_treino': np.nan})   # Mesma linha do arquivo, trocando método e métricas
    previsoes.to_csv(os.path.join(pasta_previsoes, 'previsao_' + os.path.basename(info.arquivo)), index=False)   # Salva no Drive
    if n % 20 == 0 or n == len(info_teste):                  # Progresso a cada 20 arquivos
        print(f'{n}/{len(info_teste)} arquivos de teste avaliados')

tabela_mc_lstm = pd.DataFrame(linhas)                        # Todas as métricas
tabela_mc_lstm.to_csv(os.path.join(pasta_res, 'metricas_MC_todos_LSTM.csv'), index=False)   # Salva (= metricas_MC_todos.csv do MATLAB)

# --- Resumo: média e desvio padrão (mesmas colunas do metricas_MC_resumo.csv do MATLAB) ---
variaveis = ['rmse_pct', 'mae_pct', 'erro_max_pct', 'tempo_exec_s', 'convergiu', 'frac_dentro_3sigma']   # Mesmas do groupsummary do MATLAB
grupos = tabela_mc_lstm.groupby(['perfil', 'cenario_sensor', 'Soc0_est_pct', 'metodo'], dropna=False)   # dropna=False: mantém o Soc0_est_pct vazio
resumo_mc_lstm = grupos.size().rename('GroupCount').to_frame()   # Número de linhas por grupo
for v in variaveis:                                          # Média e desvio de cada métrica
    resumo_mc_lstm['mean_' + v] = grupos[v].mean()           # Média
    resumo_mc_lstm['std_' + v] = grupos[v].std()             # Desvio padrão amostral (N-1), igual ao std do MATLAB
resumo_mc_lstm = resumo_mc_lstm.reset_index()                # Grupos viram colunas
resumo_mc_lstm.to_csv(os.path.join(pasta_res, 'metricas_MC_resumo_LSTM.csv'), index=False)   # Salva

# --- Variação entre sementes de treino (média dos 20 sorteios para cada semente) ---
por_semente = tabela_mc_lstm[tabela_mc_lstm.metodo == nome_metodo].groupby(['perfil', 'cenario_sensor', 'semente_treino'])['rmse_pct'].mean().unstack('semente_treino')   # RMSE médio por semente
por_semente.to_csv(os.path.join(pasta_res, 'metricas_LSTM_por_semente_treino.csv'))   # Salva

print(f'\nLSTM (sequência): {n_ocultos_escolhido} neurônios, taxa {taxa_escolhida:g}, fração com erro {fracao_escolhida:g} — média ± desvio')   # Título
print(f'{"Perfil":10s} {"Sensor":10s} {"Método":20s} {"RMSE (%)":>16s} {"MAE (%)":>16s} {"Erro máx. (%)":>16s} {"Conv.":>6s}')   # Cabeçalho
for r in resumo_mc_lstm.itertuples(index=False):             # Uma linha por perfil × cenário × método
    print(f'{r.perfil:10s} {r.cenario_sensor:10s} {r.metodo:20s} {r.mean_rmse_pct:7.3f} ± {r.std_rmse_pct:6.3f} {r.mean_mae_pct:7.3f} ± {r.std_mae_pct:6.3f}'
          f' {r.mean_erro_max_pct:7.3f} ± {r.std_erro_max_pct:6.3f} {r.mean_convergiu:6.2f}')   # Valores
print('\nRMSE médio (%) por semente de treino:')             # Efeito da semente de treino
print(por_semente.round(3).to_string())                      # Tabela

## 11. Figuras de conferência

1. RMSE de validação × número de neurônios, × taxa de aprendizado e × erros de sensor no treino (as escolhas da célula 8).
2. Curvas de treino das 10 sementes: se a validação piora enquanto o treino melhora, é *overfitting* (a parada antecipada corta ali).
3. SoC de referência × LSTM no sorteio r1 do caso nominal de cada perfil (linha = comitê; faixa = menor e maior semente) e o erro.

As figuras do artigo e dos slides saem depois no MATLAB (L3), a partir da pasta `previsoes`, com o `aplicar_estilo_artigo`.

In [ ]:
# ===== 11. FIGURAS DE CONFERÊNCIA =====
fig, (ax_a, ax_b, ax_c) = plt.subplots(1, 3, figsize=(15, 3.5))   # Figura 1: escolha dos neurônios (A), da taxa (B) e dos erros de sensor (C)
ax_a.plot(tabela_neuronios['n_ocultos'], tabela_neuronios['rmse_val_pct'], 'o-')   # RMSE de validação × neurônios
ax_a.axvline(n_ocultos_escolhido, color='gray', linestyle='--', label='escolhido')   # Marca o escolhido
ax_a.set_xscale('log', base=2); ax_a.set_xticks(lista_n_ocultos); ax_a.set_xticklabels(lista_n_ocultos)   # Eixo 16, 32, 64
ax_a.set_xlabel('Neurônios da LSTM'); ax_a.set_ylabel('RMSE de validação (%)')   # Eixos
ax_a.set_title(f'A: neurônios (taxa {taxa_base:g})'); ax_a.legend(); ax_a.grid(True, alpha=0.3)   # Título e grade
ax_b.plot(tabela_taxas['taxa_aprendizado'], tabela_taxas['rmse_val_pct'], 'o-')   # RMSE de validação × taxa
ax_b.axvline(taxa_escolhida, color='gray', linestyle='--', label='escolhida')   # Marca a escolhida
ax_b.set_xscale('log'); ax_b.set_xlabel('Taxa de aprendizado inicial'); ax_b.set_ylabel('RMSE de validação (%)')   # Eixos
ax_b.set_title(f'B: taxa ({n_ocultos_escolhido} neurônios)'); ax_b.legend(); ax_b.grid(True, alpha=0.3)   # Título e grade
ax_c.bar([f'{f:g}' for f in tabela_fracoes['fracao_com_erro']], tabela_fracoes['rmse_val_pct'])   # RMSE de validação × fração com erro
ax_c.set_xlabel('Fração das sequências de treino com erro de sensor'); ax_c.set_ylabel('RMSE de validação (%)')   # Eixos
ax_c.set_title(f'C: erros de sensor (escolhida: {fracao_escolhida:g})'); ax_c.grid(True, alpha=0.3)   # Título e grade
fig.tight_layout(); fig.savefig(os.path.join(pasta_figuras, '01_escolha_hiperparametros.png'), dpi=150); plt.show()   # Salva e mostra

fig, ax = plt.subplots(figsize=(7, 4))                       # Figura 2: curvas de treino da rede final
for semente in sementes_treino:                              # Uma cor por semente
    h = historicos[(n_ocultos_escolhido, taxa_escolhida, fracao_escolhida, semente)]   # Curva desta semente
    linha, = ax.plot(h['epoca'], h['rmse_val_pct'], label=f'validação s{semente}')   # Validação (linha cheia)
    ax.plot(h['epoca'], h['rmse_treino_pct'], '--', color=linha.get_color(), alpha=0.6)   # Treino (tracejada, mesma cor)
ax.set_yscale('log'); ax.set_xlabel('Época'); ax.set_ylabel('RMSE (%)')   # Rótulos
ax.set_title('Treino (tracejado) e validação (cheio)'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)   # Título e grade
fig.tight_layout(); fig.savefig(os.path.join(pasta_figuras, '02_curvas_treino.png'), dpi=150); plt.show()   # Salva e mostra

exemplos = info_teste[(info_teste.cenario_sensor == 'nominal') & (info_teste.semente == 1)]   # Sorteio r1, caso nominal, de cada perfil
fig, eixos = plt.subplots(2, len(exemplos), figsize=(6 * len(exemplos), 6), squeeze=False, sharex='col')   # Figura 3: SoC e erro
for j, info in enumerate(exemplos.itertuples(index=False)):  # Um perfil por coluna
    p = pd.read_csv(os.path.join(pasta_previsoes, 'previsao_' + os.path.basename(info.arquivo)))   # Previsões salvas
    colunas_s = [f'Soc_lstm_s{s}' for s in sementes_treino]  # Colunas das sementes
    eixos[0, j].plot(p.t_min, 100 * p.Soc, 'k', label='Referência')   # SoC de referência
    eixos[0, j].plot(p.t_min, 100 * p.Soc_lstm_media, 'C0', label='LSTM (comitê = média das sementes)')   # Comitê
    eixos[0, j].fill_between(p.t_min, 100 * p[colunas_s].min(axis=1), 100 * p[colunas_s].max(axis=1), color='C0', alpha=0.2)   # Faixa das sementes
    eixos[0, j].set_title(f'{info.perfil}, nominal, r1'); eixos[0, j].set_ylabel('SoC (%)'); eixos[0, j].legend(); eixos[0, j].grid(True, alpha=0.3)   # Rótulos
    eixos[1, j].plot(p.t_min, 100 * (p.Soc_lstm_media - p.Soc), 'C0')   # Erro do comitê (p.p.)
    eixos[1, j].axhspan(-100 * limiar_conv, 100 * limiar_conv, color='gray', alpha=0.15)   # Faixa de ±2 p.p.
    eixos[1, j].set_xlabel('Tempo (min)'); eixos[1, j].set_ylabel('Erro (p.p.)'); eixos[1, j].grid(True, alpha=0.3)   # Rótulos
fig.tight_layout(); fig.savefig(os.path.join(pasta_figuras, '03_soc_teste_nominal_r1.png'), dpi=150); plt.show()   # Salva e mostra

## 12. Checagens de integridade

Conferências automáticas de que a comparação é justa. Se alguma falhar, o notebook para com a mensagem do problema.
1. A normalização usou **só** as amostras de treino.
2. Nenhum arquivo de teste está no treino ou na validação.
3. **A estimativa no instante $k$ só usa amostras até $k$:** a rede estima o perfil cortado em 100 s e o perfil inteiro, e os primeiros 100 valores precisam ser iguais.
4. Nenhuma métrica vazia, e uma linha por arquivo × semente (+ comitê).

In [ ]:
# ===== 12. CHECAGENS DE INTEGRIDADE =====
assert np.all(normalizador.n_samples_seen_ == n_amostras_treino), 'A normalização não usou só o treino'   # 1
nomes_teste = {os.path.basename(a) for a in arq_teste}       # Nomes dos arquivos de teste
assert nomes_teste.isdisjoint({os.path.basename(a) for a in arq_treino + arq_validacao}), 'Arquivo de teste no treino/validação'   # 2

arq_c = info_teste.iloc[0]['arquivo']                        # 3. Um arquivo de teste qualquer
k_corte = 100                                                # Corta em 100 s
x_inteiro = entradas_arquivo(arq_c, dispositivo)             # Perfil inteiro
inteiro = prever_seq(modelos[sementes_treino[0]], x_inteiro)[0][:k_corte]   # Primeiros 100 valores, com o perfil inteiro
cortado = prever_seq(modelos[sementes_treino[0]], x_inteiro[:, :k_corte])[0]   # Só com as primeiras 100 amostras
assert np.allclose(inteiro, cortado, atol=1e-5), 'A estimativa usou amostras futuras'   # 3

assert not tabela_mc_lstm[['rmse_pct', 'mae_pct', 'erro_max_pct']].isna().any().any(), 'Há métrica vazia'   # 4
n_metodos = len(sementes_treino) + (1 if len(sementes_treino) > 1 else 0)   # Redes individuais + comitê
assert len(tabela_mc_lstm) == len(arq_teste) * n_metodos, 'Número de linhas errado'                    # 4
print('Todas as checagens passaram.')                        # Tudo certo
print(f'Resultados em: {pasta_res}')                         # Onde estão os arquivos

## 13. Próximo passo (L3)

1. Baixe `metricas_MC_resumo_LSTM.csv` e a pasta `previsoes` para `resultados ekf` no PC.
2. Na L3, o MATLAB junta a LSTM com a CC e o EKF (`metricas_MC_resumo.csv`) na tabela final e faz as figuras CC × EKF × LSTM com o `aplicar_estilo_artigo`.
3. Envie para o Claude: a saída das células 8, 9, 10 e 12 (texto) e as 3 figuras de `figuras/`.